<a href="https://colab.research.google.com/github/Reshsajee/my-project-resh/blob/main/Copy_of_time_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
image_dir = "/content/drive/MyDrive/mushroom/mush/data/mh-dataset/images"
label_dir = "/content/drive/MyDrive/mushroom/mush/data/mh-dataset/labels"

In [ ]:
IMG_SIZE = 224

X = []
y = []

image_files = sorted(os.listdir(image_dir))

for img_file in image_files:
    img_path = os.path.join(image_dir, img_file)

    label_file = os.path.splitext(img_file)[0] + ".txt"
    label_path = os.path.join(label_dir, label_file)

    if not os.path.exists(label_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_input(img)   # ✅ EfficientNet preprocessing
    X.append(img)

    with open(label_path, 'r') as f:
        label = f.readline().split()[0]
        y.append(int(label))

X = np.array(X)
y = np.array(y)

print("Data Loaded:", X.shape, y.shape)

In [ ]:
# 1️⃣ Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2️⃣ One-hot encoding
y_train = to_categorical(y_train, 3)
y_test = to_categorical(y_test, 3)

# 3️⃣ Compute class weights to fix imbalance
class_weights_raw = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)
class_weights = dict(enumerate(class_weights_raw))
print("Class Weights:", class_weights)

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

# Fit augmentation to training data
datagen.fit(X_train)

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze all layers initially
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# Classification head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(3, activation='softmax')(x)

# Complete model
model = models.Model(inputs=base_model.input, outputs=output)

# Compile
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=16),
    validation_data=(X_test, y_test),
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stop]
)

In [ ]:
model.save("/content/drive/MyDrive/efficientnet_mushroom_stagee.h5")

print("Model saved successfully!")

In [ ]:
# Unfreeze last 50 layers
for layer in base_model.layers[-50:]:
    layer.trainable = True

# Recompile with low learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Fine-tuning with early stopping
history_finetune = model.fit(
    datagen.flow(X_train, y_train, batch_size=16),
    validation_data=(X_test, y_test),
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stop]
)

In [ ]:
model.save("/content/drive/MyDrive/efficientnet_mushroom_stagemine.h5")

print("Model saved successfully!")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred_classes))
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred_classes))

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from tensorflow.keras.applications.efficientnet import preprocess_input
from google.colab import files

# Upload image
uploaded = files.upload()

# Get uploaded file name
img_path = list(uploaded.keys())[0]

# Prediction function
def predict_image(img_path):
    # Read image
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))

    # Preprocess for EfficientNet
    img_input = preprocess_input(img)
    img_input = np.expand_dims(img_input, axis=0)

    # Predict
    pred = model.predict(img_input)
    class_idx = np.argmax(pred)

    class_names = ['Pinhead', 'Growing', 'Mature']

    print(f"✅ Predicted Class: {class_names[class_idx]}")
    print(f"Confidence: {np.max(pred):.4f}")

    # Show image
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Predicted: {class_names[class_idx]}")
    plt.axis('off')
    plt.show()

# Run prediction
predict_image(img_path)

In [ ]:
from datetime import datetime

def extract_timestamp(img_name):
    try:
        # Example: IMG_20240301_080000.jpg
        parts = img_name.split('_')

        date_part = parts[1]              # 20240301
        time_part = parts[2].split('.')[0]  # 080000

        full_time = date_part + time_part

        timestamp = datetime.strptime(full_time, "%Y%m%d%H%M%S")

        return timestamp

    except Exception as e:
        print("Error in:", img_name)
        return None

In [ ]:
test_name = "IMG_20240301_080000.jpg"
print(extract_timestamp(test_name))

In [ ]:
import os
import cv2
import numpy as np
from datetime import datetime

dataset_path = "/content/drive/MyDrive/R11/MG"

X = []
y = []

# STEP 1: timestamp function (reuse)
def extract_timestamp(img_name):
    parts = img_name.split('_')
    date_part = parts[1]
    time_part = parts[2].split('.')[0]
    full_time = date_part + time_part
    return datetime.strptime(full_time, "%Y%m%d%H%M%S")

# STEP 2: loop through dataset
for folder in sorted(os.listdir(dataset_path)):
    folder_path = os.path.join(dataset_path, folder)

    if not os.path.isdir(folder_path):
        continue

    images = sorted(os.listdir(folder_path))

    # 🔥 Extract timestamps for all images in folder
    timestamps = [extract_timestamp(img) for img in images]

    # 🔥 Harvest time = LAST image time
    harvest_time = max(timestamps)

    for img_name, current_time in zip(images, timestamps):
        img_path = os.path.join(folder_path, img_name)

        # Load image
        img = cv2.imread(img_path)
        img = cv2.resize(img, (224, 224))
        img = img / 255.0

        X.append(img)

        # 🔥 Calculate remaining time in HOURS
        remaining_time = (harvest_time - current_time).total_seconds() / 3600.0
        y.append(remaining_time)

# Convert to numpy
X = np.array(X)
y = np.array(y)

print("🎉 Dataset Loaded Successfully!")
print("X shape:", X.shape)
print("y shape:", y.shape)

# 🔍 Check few values
print("Sample time labels:", y[:10])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze initially
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)
output = Dense(1, activation='linear')(x)

model_time = Model(inputs=base_model.input, outputs=output)

model_time.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mae']
)

model_time.summary()

In [ ]:
history = model_time.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=8
)

In [ ]:
from google.colab import files
import cv2
import numpy as np

def predict_time(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    time_pred = model_time.predict(img)[0][0]

    # 🔥 Convert hours → days + hours
    days = int(time_pred // 24)
    hours = int(time_pred % 24)

    print(f"⏳ Time Remaining: {days} days and {hours} hours")

# Upload and predict
uploaded = files.upload()
img_name = list(uploaded.keys())[0]

predict_time(img_name)

In [ ]:
from google.colab import files
import cv2
import numpy as np
from tensorflow.keras.applications.efficientnet import preprocess_input
import matplotlib.pyplot as plt

# 🔹 Stage class names
class_names = ['Pinhead', 'Growing', 'Mature']

def predict_stage_and_time(img_path):
    # Read image
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, (224, 224))

    # -------- STAGE PREDICTION --------
    img_stage = preprocess_input(img_resized.copy())
    img_stage = np.expand_dims(img_stage, axis=0)

    stage_pred = model.predict(img_stage)
    stage_idx = np.argmax(stage_pred)
    stage_name = class_names[stage_idx]

    # -------- TIME PREDICTION --------
    img_time = img_resized / 255.0
    img_time = np.expand_dims(img_time, axis=0)

    time_pred = model_time.predict(img_time)[0][0]

    # Convert to days + hours
    days = int(time_pred // 24)
    hours = int(time_pred % 24)

    # -------- OUTPUT --------
    print(f"🌱 Stage: {stage_name}")
    print(f"⏳ Time Remaining: {days} days and {hours} hours")

    # Show image
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{stage_name} | {days}d {hours}h left")
    plt.axis('off')
    plt.show()


# 🔥 Upload and test
uploaded = files.upload()
img_name = list(uploaded.keys())[0]

predict_stage_and_time(img_name)

In [ ]:
from google.colab import files
import cv2
import numpy as np
from tensorflow.keras.applications.efficientnet import preprocess_input
import matplotlib.pyplot as plt

# Class labels
class_names = ['Pinhead', 'Growing', 'Mature']

def predict_stage_with_fixed_time(img_path):
    # Read image
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, (224, 224))

    # Preprocess for stage model
    img_input = preprocess_input(img_resized)
    img_input = np.expand_dims(img_input, axis=0)

    # Predict stage
    pred = model.predict(img_input)
    stage_idx = np.argmax(pred)
    stage_name = class_names[stage_idx]

    # 🔥 Rule-based time mapping
    if stage_name == "Pinhead":
        time_text = "3 days remaining to harvest"
    elif stage_name == "Growing":
        time_text = "1 day remaining to harvest"
    else:
        time_text = "Ready to harvest"

    # Output
    print(f"🌱 Stage: {stage_name}")
    print(f"⏳ {time_text}")

    # Show image
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{stage_name} | {time_text}")
    plt.axis('off')
    plt.show()


# Upload and test
uploaded = files.upload()
img_name = list(uploaded.keys())[0]

predict_stage_with_fixed_time(img_name)